In [1]:
# imports
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# LOAD DELIVERABLES (ALREADY LAGGED)
df = pd.read_parquet("data/fx_system_df_lagged.parquet")
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# CONFIG
SHORT_WINDOW = 50
LONG_WINDOW = 200
H = 21
MIN_CROSS_SECTION = 5

# FX MAP
FX_TICKERS = {
    "AUD": {"ticker": "AUDUSD=X", "invert": False},
    "CAD": {"ticker": "CADUSD=X", "invert": True}, 
    "CHF": {"ticker": "CHFUSD=X", "invert": True},
    "EUR": {"ticker": "EURUSD=X", "invert": False},
    "GBP": {"ticker": "GBPUSD=X", "invert": False},
    "JPY": {"ticker": "JPYUSD=X", "invert": True}, 
    "NZD": {"ticker": "NZDUSD=X", "invert": False}
}

# BUILD VALUE (SINGLE DEFINITION)
def build_value_signal(df, fx_map, short, long):

    value = pd.DataFrame(index=df.index)

    for ccy in fx_map.keys():
        price = df[f"{ccy}_price"]
        log_p = np.log(price)

        ma_short = log_p.rolling(short).mean()
        ma_long  = log_p.rolling(long).mean()

        # trend proxy
        trend = ma_short - ma_long

        # flip → losers = cheap
        value[ccy] = -trend

    return value

value_signal = build_value_signal(df, FX_TICKERS, SHORT_WINDOW, LONG_WINDOW)

# CROSS-SECTIONAL NORMALIZATION (FIXED)
value_signal = value_signal.sub(value_signal.mean(axis=1), axis=0)
value_signal = value_signal.div(value_signal.std(axis=1), axis=0)

# ALIGN + COVERAGE FILTER
common_idx = value_signal.index.intersection(returns_df.index)
value_signal = value_signal.loc[common_idx]
returns_df = returns_df.loc[common_idx]

valid_counts = value_signal.notna().sum(axis=1)
value_signal = value_signal[valid_counts >= MIN_CROSS_SECTION]
returns_df = returns_df.loc[value_signal.index]

# FORWARD RETURNS
def compute_forward_returns(returns_df, H):
    return returns_df.rolling(H).sum().shift(-H)

future_returns = compute_forward_returns(returns_df, H)

common_idx = value_signal.index.intersection(future_returns.index)
value_signal = value_signal.loc[common_idx]
future_returns = future_returns.loc[common_idx]
returns_df = returns_df.loc[common_idx]

# IC SERIES
def compute_ic_series(signal, future_returns):
    ic_vals = []

    for date in signal.index:
        s = signal.loc[date]
        r = future_returns.loc[date]

        valid = s.notna() & r.notna()

        if valid.sum() < MIN_CROSS_SECTION:
            ic_vals.append(np.nan)
            continue

        if s[valid].std() == 0:
            ic_vals.append(np.nan)
            continue

        ic_vals.append(spearmanr(s[valid], r[valid]).statistic)

    return pd.Series(ic_vals, index=signal.index)

ic_series = compute_ic_series(value_signal, future_returns).dropna()

# IC DIAGNOSTICS
expanding_ic_ir = ic_series.expanding().mean() / ic_series.expanding().std()
rolling_ic_ir = ic_series.rolling(252).mean() / ic_series.rolling(252).std()

# PORTFOLIO (SIMPLE, NO STACKING)
def build_positions(signal):
    rank = signal.rank(axis=1, pct=True)

    long = (rank >= 0.67).astype(float)
    short = (rank <= 0.33).astype(float)

    pos = long - short
    pos = pos.sub(pos.mean(axis=1), axis=0)

    gross = pos.abs().sum(axis=1).replace(0, np.nan)
    pos = pos.div(gross, axis=0)

    return pos

positions = build_positions(value_signal)

pnl = (positions.shift(1) * returns_df).sum(axis=1)

sharpe = pnl.mean() / pnl.std() * np.sqrt(252)

# OUTPUT
results = {
    "IC Mean": ic_series.mean(),
    "IC Median": ic_series.median(),
    "IC IR": ic_series.mean() / ic_series.std(),
}

results_df = pd.DataFrame([results])

# SAVE DELIVERABLES
value_signal.to_parquet("data/value_signal.parquet")
ic_series.to_frame("IC").to_parquet("data/value_ic_series.parquet")
pnl.to_frame("pnl").to_parquet("data/value_pnl.parquet")
results_df.to_parquet("data/value_summary.parquet")

In [2]:
results_df

,IC Mean,IC Median,IC IR
0,0.035846,0.071429,0.064176
